# Phase 2: Discrete Multi-Stage Mutation & Leakage Verifiers (v2)
### IPC2BNS-Verify: Gated Checkpoints Preventing Cascading Retrieval & Generation Failures

This notebook verifies that discrete gate verifiers catch pipeline errors early:
1. **Stage 1 (Input Timeline Paradox Gate)**: Catches impossible date sequences (e.g. `fir_date < incident_date`).
2. **Stage 2 (Statutory Retrieval Scope Gate)**: Prevents regime cross-contamination (e.g. retrieving modern BNS bare-acts for a 1990 legacy crime).
3. **Stage 3 (Concordance & Repeal Veto Gate)**: Vetoes hallucinated BNS sections and force-mapped repealed provisions.
4. **Stage 4 (Entity Grounding & Penal Invariant Gate)**: Ensures prison sentencing and penal durations match bare-act bounds.

In [ ]:
# ==============================================================================
# STEP 0: GOOGLE COLAB & GOOGLE DRIVE INITIALIZATION (RUN THIS FIRST)
# ==============================================================================
import os, sys
from pathlib import Path

try:
    import google.colab
    IN_COLAB = True
    print("[Colab] Detected Google Colab environment. Mounting Google Drive...")
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    
    # Check all possible Drive directory variations
    drive_candidates = [
        Path('/content/drive/MyDrive/NLP_rspaper'),
        Path('/content/drive/MyDrive/NLP-rspaper'),
        Path('/content/drive/MyDrive/research paper/NLP_rs'),
        Path('/content/drive/MyDrive/NLP_rs'),
        Path.cwd()
    ]
    
    found_dir = None
    for cand in drive_candidates:
        if (cand / 'code' / 'src').exists():
            found_dir = cand
            break
            
    if not found_dir:
        # Search inside MyDrive
        print("[Colab] Searching Google Drive for project folder...")
        for match in Path('/content/drive/MyDrive').glob('**/code/src'):
            found_dir = match.parent.parent
            break
            
    if found_dir:
        os.chdir(found_dir)
        print(f"[Colab] Working directory set to: {os.getcwd()}")
        sys.path.insert(0, str(found_dir / 'code'))
    else:
        print("[Colab] Warning: Could not locate project directory in Google Drive. Using:", os.getcwd())
        
    if Path("requirements.txt").exists():
        print("[Colab] Installing dependencies from requirements.txt...")
        get_ipython().system("pip install -q -r requirements.txt")
except ImportError:
    IN_COLAB = False
    print("[Local] Running in local environment:", os.getcwd())
    root_candidates = [Path.cwd(), Path.cwd().parent, Path.cwd() / 'code']
    for p in root_candidates:
        if (p / 'code' / 'src').exists():
            sys.path.insert(0, str(p / 'code'))
            break
        elif (p / 'src').exists():
            sys.path.insert(0, str(p))
            break

print("[Setup Complete] sys.path[0]:", sys.path[0])
print("[Setup Complete] Current working directory:", os.getcwd())


## 1. Import Stage Leakage Verifier

In [ ]:
# --- Self-Healing Colab Environment Bootstrap ---
import os, sys
from pathlib import Path

# 1. Check & insert paths where 'src' is located
candidates = [
    Path.cwd() / 'code',
    Path.cwd(),
    Path('/content/drive/MyDrive/NLP_rspaper/code'),
    Path('/content/drive/MyDrive/NLP_rspaper'),
    Path('/content/drive/MyDrive/NLP-rspaper/code'),
    Path('/content/drive/MyDrive/NLP-rspaper'),
    Path('/content/drive/MyDrive/research paper/NLP_rs/code'),
    Path('/content/drive/MyDrive/research paper/NLP_rs')
]

for p in candidates:
    if (p / 'src').exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))
        if os.getcwd() != str(p.parent if p.name == 'code' else p):
            try:
                os.chdir(str(p.parent if p.name == 'code' else p))
            except Exception:
                pass
        break
# ------------------------------------------------

from src.verifier.stage_leakage_verifier import StageLeakageVerifier

verifier = StageLeakageVerifier()
print("StageLeakageVerifier initialized successfully!")

## 2. Test Mutation Injections Across All 4 Stages

In [ ]:
# Stage 1: Timeline Paradox Mutation
r1 = verifier.verify_stage1_timeline({"incident_date": "2024-08-10", "fir_date": "2024-06-01"})
print("Stage 1 Result (Paradox Check):", r1["passed"], "| Reason:", r1.get("reason"))

# Stage 2: Regime Leakage Mutation
r2 = verifier.verify_stage2_retrieval_scope(
    target_regime="LEGACY_IPC",
    retrieved_chunks=[{"act": "BNS_2023", "section": "103"}]
)
print("Stage 2 Result (Scope Check)  :", r2["passed"], "| Reason:", r2.get("reason"))

# Stage 3: Concordance / Repeal Citation Veto
r3 = verifier.verify_stage3_citations(
    citations=["IPC §124A"],
    is_post_july_offence=True
)
print("Stage 3 Result (Repeal Check) :", r3["passed"], "| Reason:", r3.get("reason"))

# Stage 4: Grounding & Penal Invariant Mutation
r4 = verifier.verify_stage4_grounding(
    generated_claim="Punishment is 20 years rigorous imprisonment",
    statutory_text="Whoever cheats shall be punished with imprisonment for a term which may extend to seven years."
)
print("Stage 4 Result (Penal Invariant):", r4["passed"], "| Reason:", r4.get("reason"))

## 3. Verify Pytest Verifier Suite

In [ ]:
!pytest code/tests/test_stage_verifier.py code/tests/test_verifier.py -v